In [1]:
import duckdb
import os

os.chdir('..')
conn = duckdb.connect("data/processed/quickbite.duckdb")
print("Connected to warehouse")
print(conn.execute("SHOW TABLES").fetchdf())

Connected to warehouse
             name
0  dim_conditions
1     dim_courier
2        dim_date
3    dim_location
4  dim_restaurant
5     fact_orders


In [2]:
q1 = """
WITH weekly_stats AS (
    SELECT 
        DATE_TRUNC('week', d.order_date) AS week_start,
        COUNT(*) AS total_orders,
        SUM(CASE WHEN f.is_on_time = 1 THEN 1 ELSE 0 END) AS on_time_orders,
        ROUND(AVG(f.delivery_minutes), 1) AS avg_delivery_min
    FROM fact_orders f
    JOIN dim_date d ON f.date_id = d.date_id
    GROUP BY week_start
)
SELECT 
    week_start,
    total_orders,
    on_time_orders,
    avg_delivery_min,
    ROUND(100.0 * on_time_orders / total_orders, 1) AS sla_pct
FROM weekly_stats
ORDER BY week_start;
"""
result_q1 = conn.execute(q1).fetchdf()
print(result_q1)

  week_start  total_orders  on_time_orders  avg_delivery_min  sla_pct
0 2022-02-07          2088          1531.0              25.5     73.3
1 2022-02-14          3403          2239.0              27.3     65.8
2 2022-02-28          5647          3969.0              26.3     70.3
3 2022-03-07          6620          4735.0              25.9     71.5
4 2022-03-14          6484          4367.0              27.0     67.4
5 2022-03-21          5600          3899.0              26.3     69.6
6 2022-03-28          6615          4761.0              26.0     72.0
7 2022-04-04          2684          1763.0              27.3     65.7


In [5]:
q2 = """
SELECT 
    'Traffic: ' || c.traffic AS factor,
    COUNT(*) AS order_count,
    ROUND(AVG(f.delivery_minutes), 1) AS avg_delivery_min,
    ROUND(100.0 * SUM(f.is_on_time) / COUNT(*), 1) AS sla_pct
FROM fact_orders f
JOIN dim_conditions c ON f.condition_id = c.condition_id
GROUP BY c.traffic

UNION ALL

SELECT 
    'Weather: ' || c.weather AS factor,
    COUNT(*),
    ROUND(AVG(f.delivery_minutes), 1),
    ROUND(100.0 * SUM(f.is_on_time) / COUNT(*), 1)
FROM fact_orders f
JOIN dim_conditions c ON f.condition_id = c.condition_id
GROUP BY c.weather

UNION ALL

SELECT 
    CASE WHEN f.is_peak_hour THEN 'Peak Hour: Yes' ELSE 'Peak Hour: No' END,
    COUNT(*),
    ROUND(AVG(f.delivery_minutes), 1),
    ROUND(100.0 * SUM(f.is_on_time) / COUNT(*), 1)
FROM fact_orders f
GROUP BY f.is_peak_hour

UNION ALL

SELECT 
    'Vehicle: ' || f.vehicle_type,
    COUNT(*),
    ROUND(AVG(f.delivery_minutes), 1),
    ROUND(100.0 * SUM(f.is_on_time) / COUNT(*), 1)
FROM fact_orders f
GROUP BY f.vehicle_type

ORDER BY avg_delivery_min DESC;
"""
result_q2 = conn.execute(q2).fetchdf()
print(result_q2)

                       factor  order_count  avg_delivery_min  sla_pct
0                Traffic: Jam        12420              31.2     48.9
1              Peak Hour: Yes        13946              30.7     51.2
2                Weather: Fog         6668              29.1     55.4
3             Weather: Cloudy         6569              29.0     55.6
4         Vehicle: motorcycle        22998              27.7     65.7
5               Traffic: High         3866              27.2     66.5
6             Traffic: Medium         9571              26.8     67.1
7              Weather: Windy         6467              26.2     73.1
8             Weather: Stormy         6606              26.0     73.8
9         Weather: Sandstorms         6496              26.0     73.7
10           Vehicle: scooter        13031              24.6     75.1
11  Vehicle: electric_scooter         3112              24.5     76.3
12              Peak Hour: No        25195              24.0     79.9
13             Weath

In [6]:
q3 = """
WITH courier_stats AS (
    SELECT 
        f.courier_id,
        c.courier_age,
        c.courier_rating,
        c.vehicle_condition,
        COUNT(*) AS total_orders,
        ROUND(AVG(f.delivery_minutes), 1) AS avg_delivery_min,
        ROUND(100.0 * SUM(f.is_on_time) / COUNT(*), 1) AS on_time_pct
    FROM fact_orders f
    JOIN dim_courier c ON f.courier_id = c.courier_id
    GROUP BY f.courier_id, c.courier_age, c.courier_rating, c.vehicle_condition
    HAVING COUNT(*) >= 20
),
ranked AS (
    SELECT 
        *,
        RANK() OVER (ORDER BY on_time_pct DESC, avg_delivery_min ASC) AS top_rank,
        RANK() OVER (ORDER BY on_time_pct ASC, avg_delivery_min DESC) AS bottom_rank
    FROM courier_stats
)
SELECT 
    courier_id, 
    courier_age, 
    courier_rating, 
    total_orders,
    avg_delivery_min,
    on_time_pct,
    'TOP' AS performance_group
FROM ranked WHERE top_rank <= 10

UNION ALL

SELECT 
    courier_id, 
    courier_age, 
    courier_rating, 
    total_orders,
    avg_delivery_min,
    on_time_pct,
    'BOTTOM' AS performance_group
FROM ranked WHERE bottom_rank <= 10

ORDER BY performance_group, on_time_pct DESC;
"""
result_q3 = conn.execute(q3).fetchdf()
print(result_q3)

          courier_id  courier_age  courier_rating  total_orders  \
0      MUMRES18DEL01           25             4.6            50   
1      VADRES01DEL01           31             4.4            56   
2      HYDRES20DEL01           28             4.6            55   
3   RANCHIRES13DEL01           38             4.7            57   
4     CHENRES17DEL01           32             5.0            46   
5     INDORES19DEL01           35             4.7            54   
6    COIMBRES05DEL02           39             4.9            50   
7     CHENRES03DEL01           28             4.0            53   
8     JAPRES010DEL01           38             4.9            52   
9     CHENRES14DEL01           38             4.5            49   
10     MUMRES12DEL03           26             4.7            40   
11   COIMBRES07DEL03           24             4.7            39   
12     MUMRES12DEL02           32             4.8            53   
13    PUNERES07DEL02           37             4.7            5

In [7]:
q4 = """
SELECT 
    l.city_type,
    CASE 
        WHEN f.distance_km < 5 THEN '0-5 km (short)'
        WHEN f.distance_km < 10 THEN '5-10 km (medium)'
        WHEN f.distance_km < 20 THEN '10-20 km (long)'
        ELSE '20+ km (very long)'
    END AS distance_bucket,
    COUNT(*) AS order_count,
    ROUND(AVG(f.delivery_minutes), 1) AS avg_delivery_min,
    ROUND(100.0 * SUM(f.is_on_time) / COUNT(*), 1) AS sla_pct
FROM fact_orders f
JOIN dim_location l ON f.location_id = l.location_id
GROUP BY l.city_type, distance_bucket
ORDER BY l.city_type, avg_delivery_min DESC;
"""
result_q4 = conn.execute(q4).fetchdf()
print(result_q4)

       city_type     distance_bucket  order_count  avg_delivery_min  sla_pct
0   Metropolitan  20+ km (very long)         1322              31.0     49.9
1   Metropolitan     10-20 km (long)        13071              30.6     51.2
2   Metropolitan    5-10 km (medium)         8063              25.3     76.2
3   Metropolitan      0-5 km (short)         7652              23.1     85.6
4     Semi-Urban  20+ km (very long)            9              48.7      0.0
5     Semi-Urban    5-10 km (medium)           18              48.2      0.0
6     Semi-Urban     10-20 km (long)          100              48.2      0.0
7     Semi-Urban      0-5 km (short)           14              47.7      0.0
8          Urban     10-20 km (long)         3256              26.6     67.1
9          Urban  20+ km (very long)          339              25.6     72.9
10         Urban    5-10 km (medium)         2447              21.6     87.0
11         Urban      0-5 km (short)         2850              19.8     93.2

In [10]:
q5 = """
WITH restaurant_perf AS (
    SELECT 
        f.restaurant_id,
        r.restaurant_lat,
        r.restaurant_lng,
        COUNT(*) AS total_orders,
        ROUND(AVG(
            (EXTRACT(HOUR FROM f.time_picked) * 3600 + 
             EXTRACT(MINUTE FROM f.time_picked) * 60 + 
             EXTRACT(SECOND FROM f.time_picked))
            -
            (EXTRACT(HOUR FROM f.time_ordered) * 3600 + 
             EXTRACT(MINUTE FROM f.time_ordered) * 60 + 
             EXTRACT(SECOND FROM f.time_ordered))
        ) / 60.0, 1) AS avg_prep_min,
        ROUND(AVG(f.delivery_minutes), 1) AS avg_total_delivery_min,
        ROUND(100.0 * SUM(f.is_on_time) / COUNT(*), 1) AS sla_pct
    FROM fact_orders f
    JOIN dim_restaurant r ON f.restaurant_id = r.restaurant_id
    GROUP BY f.restaurant_id, r.restaurant_lat, r.restaurant_lng
    HAVING COUNT(*) >= 30
)
SELECT 
    restaurant_id,
    restaurant_lat,
    restaurant_lng,
    total_orders,
    avg_prep_min,
    avg_total_delivery_min,
    sla_pct
FROM restaurant_perf
ORDER BY avg_prep_min DESC
LIMIT 15;
"""
result_q5 = conn.execute(q5).fetchdf()
print(result_q5)

    restaurant_id  restaurant_lat  restaurant_lng  total_orders  avg_prep_min  \
0             289       30.885915       75.788259            37          11.5   
1              81       26.479108       80.315042            30          11.2   
2             315       30.332735       78.054222            39          11.0   
3             360       23.230791       77.437020            32          10.9   
4             224       19.875522       75.367127            32          10.9   
5              46       27.165108       78.015053            38          10.9   
6             383       15.576683       73.755750            32          10.9   
7             226       30.895204       75.822103            38          10.8   
8             373       22.515082       88.367830            30          10.8   
9             152       11.006686       76.951736           151          10.8   
10            337       27.161850       78.040165            33          10.8   
11            326       25.4

In [11]:
q6 = """
WITH festival_comparison AS (
    SELECT 
        is_festival,
        COUNT(*) AS order_count,
        ROUND(AVG(delivery_minutes), 1) AS avg_delivery_min,
        ROUND(AVG(distance_km), 1) AS avg_distance_km,
        ROUND(100.0 * SUM(is_on_time) / COUNT(*), 1) AS sla_pct
    FROM fact_orders
    GROUP BY is_festival
)
SELECT 
    is_festival,
    order_count,
    avg_delivery_min,
    avg_distance_km,
    sla_pct,
    ROUND(
        avg_delivery_min - LAG(avg_delivery_min) OVER (ORDER BY is_festival), 1
    ) AS delivery_min_diff_vs_no_festival
FROM festival_comparison
ORDER BY is_festival;
"""
result_q6 = conn.execute(q6).fetchdf()
print(result_q6)

  is_festival  order_count  avg_delivery_min  avg_distance_km  sla_pct  \
0          No        38366              26.0              9.7     71.1   
1         Yes          775              45.0             13.2      0.0   

   delivery_min_diff_vs_no_festival  
0                               NaN  
1                              19.0  


In [12]:
conn.close()
print("All KPI queries executed successfully")

All KPI queries executed successfully
